08_business_insights.ipynb

1. Project Objective
2. Dataset Scope
3. Executive KPIs
4. Operational Findings
5. Airline Findings
6. Airport Findings
7. Delay Cause Findings
8. Weather Findings
9. Key Business Problems
10. Business Recommendations
11. Final Executive Summary

# Airline Operations & Disruption Intelligence
## Business Insights

### Business Problem

Airline operations can be affected by delays, cancellations, diversions,
airport-level operational conditions, and weather.

The goal of this project is to identify the major operational patterns
and understand how weather conditions are associated with flight disruptions.

### Main Business Question

How can airline operations teams identify and understand
the major drivers of flight delays and disruptions?

In [ ]:
Step 1 — Create the notebook

Create:

08_business_insights.ipynb

This notebook should not redo your EDA. It should convert your EDA results into business conclusions.

Notebook structure
08_business_insights.ipynb


1. Project Objective
2. Dataset Scope
3. Executive KPIs
4. Operational Findings
5. Airline Findings
6. Airport Findings
7. Delay Cause Findings
8. Weather Findings
9. Key Business Problems
10. Business Recommendations
11. Final Executive Summary
1. Project Objective

Markdown cell:

# Airline Operations & Disruption Intelligence
## Business Insights


### Business Problem


Airline operations can be affected by delays, cancellations, diversions,
airport-level operational conditions, and weather.


The goal of this project is to identify the major operational patterns
and understand how weather conditions are associated with flight disruptions.


### Main Business Question


How can airline operations teams identify and understand
the major drivers of flight delays and disruptions?

Your EDA already establishes the central question as understanding flight operations, delays, cancellations, diversions, and weather impact.

2. Dataset Scope

Add:

## Dataset Scope


The analysis uses historical BTS On-Time Performance flight data
for January–March 2026.


Additional airport reference information was integrated to identify
airport locations.


Historical weather information was collected from Open-Meteo
and joined using airport, date, and scheduled departure hour.


The final dataset was then used for exploratory analysis
and business insight generation.

This is important for interviews because you can explain the project from beginning to end.

3. Executive KPIs

Load your final dataset:

import pandas as pd


file_path = r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\processed\flights_weather_enriched.csv"


df = pd.read_csv(file_path)


df["fl_date"] = pd.to_datetime(df["fl_date"], errors="coerce")


print("Rows:", len(df))
print("Date:", df["fl_date"].min(), "to", df["fl_date"].max())

Then:

total_flights = len(df)


cancelled = df["cancelled"].sum()
diverted = df["diverted"].sum()


delay_rate = df["arr_del15"].mean() * 100


print("Total flights:", total_flights)
print("Cancelled flights:", cancelled)
print("Cancellation rate:", round(cancelled / total_flights * 100, 2), "%")
print("Diverted flights:", diverted)
print("Diversion rate:", round(diverted / total_flights * 100, 2), "%")
print("Arrival delay rate:", round(delay_rate, 2), "%")
Why this matters

These are your executive-level numbers.

A manager doesn't initially need 30 charts.

They want:

How many flights?
How many were delayed?
How many were cancelled?
How many were diverted?

4. Operational Findings

Now answer:

Finding 1 — When do delays happen?

Use your existing EDA result:

hour_delay = (
    df.groupby("departure_hour")["arr_del15"]
    .mean()
    .mul(100)
)


hour_delay.sort_values(ascending=False).head(5)

Then identify the highest hour:

highest_delay_hour = hour_delay.idxmax()
highest_delay_rate = hour_delay.max()


print("Highest delay hour:", highest_delay_hour)
print("Delay rate:", round(highest_delay_rate, 2), "%")

Write your conclusion after seeing the result:

### Finding


The highest arrival delay rate occurred during ______.


### Business Impact


This indicates that certain departure periods experienced
greater operational disruption during the study period.


### Recommendation


Operations teams could monitor high-delay departure periods
more closely and investigate whether airport congestion,
aircraft turnaround, or accumulated delays contribute to the pattern.

Don't claim congestion is the cause unless your analysis actually proves/supports it.

5. Airline Findings

Your EDA already calculates airline flight volume and arrival delay rate and applies a minimum flight-count threshold.

Use:

airline_summary = (
    df.groupby("mkt_carrier")
    .agg(
        flights=("mkt_carrier", "size"),
        cancelled=("cancelled", "sum"),
        diverted=("diverted", "sum"),
        avg_arr_delay=("arr_delay", "mean"),
        delay_rate=("arr_del15", "mean")
    )
)


airline_summary["delay_rate"] *= 100


airline_summary[
    airline_summary["flights"] >= 100
].sort_values(
    "delay_rate",
    ascending=False
).head(10)
Important

Don't say:

Airline X is bad.

Instead:

Airline X recorded the highest arrival-delay rate among airlines with at least 100 flights during the study period.

That is professional and defensible.

6. Airport Findings

Your EDA already uses a minimum flight count to avoid unstable airport comparisons.

Use:

airport_summary = (
    df.groupby("origin")
    .agg(
        flights=("origin", "size"),
        avg_arr_delay=("arr_delay", "mean"),
        delay_rate=("arr_del15", "mean")
    )
)


airport_summary["delay_rate"] *= 100


airport_summary[
    airport_summary["flights"] >= 100
].sort_values(
    "delay_rate",
    ascending=False
).head(10)

Your insight should follow:

Finding → Evidence → Business impact → Recommendation

7. Delay Causes

This is an important section because BTS already provides recorded delay causes.

Your EDA separates these from the Open-Meteo weather analysis.

delay_columns = [
    "carrier_delay",
    "weather_delay",
    "nas_delay",
    "security_delay",
    "late_aircraft_delay"
]


delay_totals = (
    df[delay_columns]
    .sum()
    .sort_values(ascending=False)
)


delay_totals

Then:

top_delay_cause = delay_totals.idxmax()
top_delay_minutes = delay_totals.max()


print("Largest recorded delay cause:", top_delay_cause)
print("Total delay minutes:", round(top_delay_minutes, 0))
Business interpretation

Don't simply say:

Weather is the biggest problem.

Remember:

weather_delay

is the BTS-recorded delay cause, while your Open-Meteo analysis uses actual weather observations. Your EDA explicitly distinguishes these two concepts.

That's a very good interview talking point.

8. Weather Findings ⭐

This is the strongest new part of your project.

Your EDA already compares:

weather category
departure delay
arrival delay
cancellation
diversion
airport weather impact

Start with:

weather_delay = (
    df.groupby("bad_weather")["arr_del15"]
    .mean()
    .mul(100)
)


weather_delay

Then:

normal_delay = weather_delay.get(0)
bad_delay = weather_delay.get(1)


print("Normal weather delay rate:", round(normal_delay, 2), "%")
print("Bad weather delay rate:", round(bad_delay, 2), "%")
print(
    "Difference:",
    round(bad_delay - normal_delay, 2),
    "percentage points"
)
Your business statement should look like:
### Weather Finding


Flights operating during bad weather had a ______% arrival
delay rate compared with ______% during normal weather.


This represents a difference of ______ percentage points.


### Business Impact


The result suggests that adverse weather conditions are
associated with higher operational disruption during the study period.


### Recommendation


Operations teams could monitor weather-sensitive airports
before scheduled departure periods and prepare for potential
operational disruption.


### Important limitation


This analysis identifies an association between weather and delays.
It does not prove that weather alone caused the delays.

That last sentence is especially important because your EDA documentation explicitly warns against treating the relationship as causation.

9. Weather vs Cancellation
weather_cancel = (
    df.groupby("bad_weather")["cancelled"]
    .mean()
    .mul(100)
)


weather_cancel

Then:

print("Normal weather cancellation rate:",
      round(weather_cancel.get(0), 2), "%")


print("Bad weather cancellation rate:",
      round(weather_cancel.get(1), 2), "%")

Don't interpret until you see the numbers.

10. Weather vs Diversion
weather_diversion = (
    df.groupby("bad_weather")["diverted"]
    .mean()
    .mul(100)
)


weather_diversion

Again:

Finding → Evidence → Impact → Recommendation

Your EDA specifically includes this comparison.

11. Key Business Problems

After running everything, make a simple table:

Priority	Business Problem	Evidence	Impact
1	Highest delay period	Your result	Operational delays
2	High-delay airport	Your result	Airport disruption
3	Major delay cause	Your result	Lost schedule reliability
4	Weather-related disruption	Your result	Delay/cancellation risk
5	High-delay airline	Your result	Lower punctuality

Do not fill this with assumptions.

Use your actual numbers.

12. Recommendations

Keep recommendations connected to findings.

If weather has a strong relationship with delays:

Monitor weather conditions at weather-sensitive airports before departure periods and prepare operational teams for potential disruption.

If certain hours have high delay rates:

Investigate operational processes during high-delay departure periods, including turnaround and accumulated delays.

If certain airports have high delay rates:

Prioritize airport-level operational review for airports with consistently high delay rates.

If one delay cause dominates:

Focus operational improvement efforts on the largest recorded delay category rather than treating all delays equally.

If cancellations are strongly associated with bad weather:

Include weather monitoring in operational disruption planning during periods of adverse conditions.

These are recommendations, not claims that your dataset has proven the exact cause.

13. Final Executive Summary

This is the most important cell in the notebook.

Use this template:

# Executive Summary


## Overall Performance


During the study period, the dataset contained ______ flights.


The overall cancellation rate was ______%, the diversion rate
was ______%, and the arrival delay rate was ______%.


## Key Operational Finding


The highest delay rate occurred during ______.


This indicates that ______ experienced greater operational
disruption during the study period.


## Airline / Airport Finding


The airline/airport with the highest delay rate after applying
the minimum flight-volume threshold was ______.


Its delay rate was ______%.


## Delay Cause


The largest recorded BTS delay category was ______,
with approximately ______ total delay minutes.


## Weather Finding


Flights during bad weather had an arrival delay rate of ______%,
compared with ______% during normal weather.


This represents a difference of ______ percentage points.


## Business Recommendation


Airline operations teams should prioritize monitoring of
high-delay periods, airports, and weather-sensitive operations.


Weather information can be used as an additional operational
signal when preparing for potential disruptions.


## Limitation


The analysis identifies patterns and associations in historical
data. It does not prove that a single factor independently
caused a flight delay or cancellation.
Your interview story becomes very strong

You can now explain the project simply:

“I started with raw BTS flight data and performed extensive data cleaning and validation. I then integrated airport reference data and historical weather from the Open-Meteo API. After validating the weather-to-flight join, I performed EDA to identify operational patterns. I then converted those findings into business insights around delays, cancellations, diversions, delay causes, airports, airlines, and weather.”